# 03 — Simple NN on Tabular Data

Küçük tabular veriyle mini model eğitimi, loss grafikleri ve sklearn karşılaştırması.

## Görev 2 — İlk ağ


In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split

# Veri oluştur
torch.manual_seed(0)
X = torch.randn(200, 3)
y = (X[:,0]*0.5 + X[:,1]*(-1.2) + 0.8*X[:,2] + 0.3 > 0).float().view(-1,1)

# Dataset + split
dataset = TensorDataset(X, y)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=16)

# Model
model = nn.Sequential(
    nn.Linear(3, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
    nn.Sigmoid()
)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Eğitim döngüsü
for epoch in range(50):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        pred = model(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    val_loss = 0
    model.eval()
    with torch.no_grad():
        for xb, yb in val_dl:
            val_loss += criterion(model(xb), yb).item()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: TrainLoss={total_loss/len(train_dl):.4f} ValLoss={val_loss/len(val_dl):.4f}")

Epoch 0: TrainLoss=0.6675 ValLoss=0.6116
Epoch 10: TrainLoss=0.1300 ValLoss=0.0988
Epoch 20: TrainLoss=0.0630 ValLoss=0.0454
Epoch 30: TrainLoss=0.0439 ValLoss=0.0258
Epoch 40: TrainLoss=0.0327 ValLoss=0.0218


## Görev 3 — sklearn karşılaştırması

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

logreg = LogisticRegression(max_iter=500)
logreg.fit(X, y.numpy().ravel())
y_pred = logreg.predict(X)
acc = accuracy_score(y, y_pred)

print('Sklearn doğruluk:', acc)
print('DL modeli fazla mı geldi? — Cevap: Küçük veri + basit ilişki, LogisticRegression genelde yeterlidir.')

Sklearn doğruluk: 0.99
DL modeli fazla mı geldi? — Cevap: Küçük veri + basit ilişki, LogisticRegression genelde yeterlidir.
